In [1]:
!pip install torchmetrics

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 983.4/983.4 kB 13.6 MB/s eta 0:00:00


In [2]:
import torch
import torch.nn as nn
import torchmetrics
from torch.utils.data import TensorDataset, DataLoader
from torchvision import datasets, transforms
import numpy as np

In [3]:
# PIL image -> (32, 32, 3) -> (3, 32, 32)

In [4]:
# CIFAR-10 datasetinin yüklənməsi
transform = transforms.ToTensor()
cifar_train = datasets.CIFAR10(root="./data", train=True,  download=True, transform=transform)
cifar_test  = datasets.CIFAR10(root="./data", train=False, download=True, transform=transform)

100%|██████████| 170M/170M [00:03<00:00, 52.6MB/s]


In [5]:
# Model A -> 8 sinif
# Model B -> 2 sinif

In [6]:
cifar_train.classes

['airplane',
 'automobile',
 'bird',
 'cat',
 'deer',
 'dog',
 'frog',
 'horse',
 'ship',
 'truck']

In [7]:
cifar_train

Dataset CIFAR10
    Number of datapoints: 50000
    Root location: ./data
    Split: Train
    StandardTransform
Transform: ToTensor()

In [8]:
# (tensor, label)

In [9]:
# df[i] -> (3, 32, 32)
# ds[i][0] -> (3, 32, 32)
# ds[i][1] ->

In [10]:
def dataset_to_tensors(ds): # tensorları (N, 3, 32, 32)
    X = torch.stack([ds[i][0] for i in range(len(ds))])   # float, [0,1]
    y = torch.tensor([ds[i][1] for i in range(len(ds))], dtype=torch.long)
    return X, y

In [11]:
X_train, y_train = dataset_to_tensors(cifar_train)
X_test,  y_test = dataset_to_tensors(cifar_test)

In [12]:
X = torch.cat([X_train, X_test]) # (50000, 3, 32, 32)
y = torch.cat([y_train, y_test]) # (10000, , 3, 32, 32)

In [13]:
# Task B sinifləri: cat(3) və dog(5)
in_B = (y == 3) | (y == 5)
X_A, y_A = X[~in_B], y[~in_B]
X_B, y_B = X[in_B], y[in_B]

# A: 8 sinif → 0..7-ə remap et
# Orijinal siniflər: 0,1,2,4,6,7,8,9 → ardıcıl 0-7
unique_A = torch.tensor([0, 1, 2, 4, 6, 7, 8, 9])
remap = {v.item(): i for i, v in enumerate(unique_A)}
y_A = torch.tensor([remap[v.item()] for v in y_A], dtype=torch.long)

# B: binary — dog(5) → 1, cat(3) → 0
y_B = (y_B == 5).to(dtype=torch.float32).view(-1, 1)

In [14]:
# 60, 000 * (2/10) = 12, 000 -> for B classes
# 60, 000 - 12, 000 = 48, 000 -> X_A

# X_A[:-7_000] => 48, 000 - 7000 = 41, 000 -> X_A training
# 7000 - 5000 = 2000 -> X_A valid
# 5000 -> X_A test

In [15]:
# Train/Test bölünməsi

# A dataset bölməsi
train_set_A = TensorDataset(X_A[:-7_000], y_A[:-7_000])
valid_set_A = TensorDataset(X_A[-7_000:-5_000], y_A[-7_000:-5_000])
test_set_A  = TensorDataset(X_A[-5_000:], y_A[-5_000:])

# B dataset bölməsi (az data — transfer learning məntiqini göstərmək üçün)
train_set_B = TensorDataset(X_B[:20],      y_B[:20])
valid_set_B = TensorDataset(X_B[20:5_000], y_B[20:5_000])
test_set_B  = TensorDataset(X_B[5_000:],   y_B[5_000:])

# DataLoader-lər
train_loader_A = DataLoader(train_set_A, batch_size=32, shuffle=True)
valid_loader_A = DataLoader(valid_set_A, batch_size=32)
test_loader_A  = DataLoader(test_set_A,  batch_size=32)

train_loader_B = DataLoader(train_set_B, batch_size=32, shuffle=True)
valid_loader_B = DataLoader(valid_set_B, batch_size=32)
test_loader_B  = DataLoader(test_set_B,  batch_size=32)

In [16]:
import torchmetrics, copy

if torch.cuda.is_available():
    device = "cuda"
elif torch.backends.mps.is_available():
    device = "mps"
else:
    device = "cpu"

In [17]:
def use_he_init(layer):
    if isinstance(layer, nn.Linear):
        nn.init.kaiming_normal_(layer.weight, nonlinearity="relu")
        nn.init.zeros_(layer.bias)

In [18]:
def evaluate_tm(model, data_loader, metric):
    model.eval()
    metric.reset()
    with torch.no_grad():
        for X_batch, y_batch in data_loader:
            X_batch, y_batch = X_batch.to(device), y_batch.to(device)
            y_pred = model(X_batch)
            metric.update(y_pred, y_batch)
    return metric.compute()

In [19]:
def train(model, optimizer, loss_fn, metric, train_loader, valid_loader, n_epochs):
    history = {"train_losses": [], "train_metrics": [], "valid_metrics": []}
    for epoch in range(n_epochs):
        total_loss = 0.0
        metric.reset()
        model.train()
        for X_batch, y_batch in train_loader:
            X_batch, y_batch = X_batch.to(device), y_batch.to(device)
            y_pred = model(X_batch)
            loss = loss_fn(y_pred, y_batch)
            total_loss += loss.item()
            loss.backward()
            optimizer.step()
            optimizer.zero_grad()
            metric.update(y_pred, y_batch)
        history["train_losses"].append(total_loss / len(train_loader))
        history["train_metrics"].append(metric.compute().item())
        history["valid_metrics"].append(
            evaluate_tm(model, valid_loader, metric).item())
        print(f"Epoch {epoch+1}/{n_epochs} | "
              f"loss: {history['train_losses'][-1]:.4f} | "
              f"train acc: {history['train_metrics'][-1]:.4f} | "
              f"val acc: {history['valid_metrics'][-1]:.4f}")
    return history

In [20]:
# Model A -8 classes
torch.manual_seed(42)

model_A = nn.Sequential(
    nn.Flatten(),
    nn.Linear(3 * 32 * 32, 256),
    nn.ReLU(),
    nn.Linear(256, 128),
    nn.ReLU(),
    nn.Linear(128, 64),
    nn.ReLU(),
    nn.Linear(64, 8)
)
model_A.to(device)

Sequential(
  (0): Flatten(start_dim=1, end_dim=-1)
  (1): Linear(in_features=3072, out_features=256, bias=True)
  (2): ReLU()
  (3): Linear(in_features=256, out_features=128, bias=True)
  (4): ReLU()
  (5): Linear(in_features=128, out_features=64, bias=True)
  (6): ReLU()
  (7): Linear(in_features=64, out_features=8, bias=True)
)

In [21]:
model_A.apply(use_he_init)

Sequential(
  (0): Flatten(start_dim=1, end_dim=-1)
  (1): Linear(in_features=3072, out_features=256, bias=True)
  (2): ReLU()
  (3): Linear(in_features=256, out_features=128, bias=True)
  (4): ReLU()
  (5): Linear(in_features=128, out_features=64, bias=True)
  (6): ReLU()
  (7): Linear(in_features=64, out_features=8, bias=True)
)

In [22]:
optimizer = torch.optim.SGD(model_A.parameters(), lr=0.005)
xentropy  = nn.CrossEntropyLoss() # 0, 1, 2, 3, 4, 5, 6, 7
accuracy  = torchmetrics.Accuracy(task="multiclass", num_classes=8).to(device)

In [23]:
history_A = train(model_A, optimizer, xentropy, accuracy,
                  train_loader_A, valid_loader_A, n_epochs=20)
history_A

Epoch 1/20 | loss: 1.6666 | train acc: 0.3733 | val acc: 0.4030
Epoch 2/20 | loss: 1.4913 | train acc: 0.4505 | val acc: 0.4490
Epoch 3/20 | loss: 1.4114 | train acc: 0.4822 | val acc: 0.4055
Epoch 4/20 | loss: 1.3610 | train acc: 0.5006 | val acc: 0.5195
Epoch 5/20 | loss: 1.3164 | train acc: 0.5171 | val acc: 0.4575
Epoch 6/20 | loss: 1.2808 | train acc: 0.5310 | val acc: 0.4895
Epoch 7/20 | loss: 1.2476 | train acc: 0.5474 | val acc: 0.4755
Epoch 8/20 | loss: 1.2223 | train acc: 0.5522 | val acc: 0.4895
Epoch 9/20 | loss: 1.1956 | train acc: 0.5635 | val acc: 0.5130
Epoch 10/20 | loss: 1.1750 | train acc: 0.5743 | val acc: 0.5145
Epoch 11/20 | loss: 1.1502 | train acc: 0.5822 | val acc: 0.4960
Epoch 12/20 | loss: 1.1320 | train acc: 0.5861 | val acc: 0.5445
Epoch 13/20 | loss: 1.1120 | train acc: 0.5965 | val acc: 0.5270
Epoch 14/20 | loss: 1.0946 | train acc: 0.6037 | val acc: 0.5310
Epoch 15/20 | loss: 1.0774 | train acc: 0.6079 | val acc: 0.5760
Epoch 16/20 | loss: 1.0614 | train

{'train_losses': [1.6666495300678306,
  1.4913185876728778,
  1.4113654524971282,
  1.3609773814678192,
  1.3164304232634545,
  1.2807896392021834,
  1.2476165764696328,
  1.2222598952454822,
  1.1955770202062432,
  1.174992635218476,
  1.1501524694810235,
  1.1320263171530738,
  1.11202424708841,
  1.094572964817425,
  1.0773686131058542,
  1.0613943100812469,
  1.0415949743828052,
  1.0327044247930024,
  1.0148990577589516,
  0.9996752935620068],
 'train_metrics': [0.3732926845550537,
  0.45051220059394836,
  0.482195109128952,
  0.5005853772163391,
  0.5171463489532471,
  0.5310487747192383,
  0.5474146604537964,
  0.5521707534790039,
  0.5634633898735046,
  0.5743414759635925,
  0.5822195410728455,
  0.5860731601715088,
  0.596487820148468,
  0.6036829352378845,
  0.6078780293464661,
  0.6151219606399536,
  0.6237804889678955,
  0.626926839351654,
  0.6319024562835693,
  0.6359268426895142],
 'valid_metrics': [0.40299999713897705,
  0.4490000009536743,
  0.40549999475479126,
  0.51

In [24]:
# Model - B
torch.manual_seed(9)

model_B = nn.Sequential(
    nn.Flatten(),
    nn.Linear(3 * 32 * 32, 256),
    nn.ReLU(),
    nn.Linear(256, 128),
    nn.ReLU(),
    nn.Linear(128, 64),
    nn.ReLU(),
    nn.Linear(64, 1)
)

model_B.to(device)

Sequential(
  (0): Flatten(start_dim=1, end_dim=-1)
  (1): Linear(in_features=3072, out_features=256, bias=True)
  (2): ReLU()
  (3): Linear(in_features=256, out_features=128, bias=True)
  (4): ReLU()
  (5): Linear(in_features=128, out_features=64, bias=True)
  (6): ReLU()
  (7): Linear(in_features=64, out_features=1, bias=True)
)

In [25]:
model_B.apply(use_he_init)

Sequential(
  (0): Flatten(start_dim=1, end_dim=-1)
  (1): Linear(in_features=3072, out_features=256, bias=True)
  (2): ReLU()
  (3): Linear(in_features=256, out_features=128, bias=True)
  (4): ReLU()
  (5): Linear(in_features=128, out_features=64, bias=True)
  (6): ReLU()
  (7): Linear(in_features=64, out_features=1, bias=True)
)

In [26]:
optimizer = torch.optim.SGD(model_B.parameters(), lr=0.005)
xentropy  = nn.BCEWithLogitsLoss()
accuracy  = torchmetrics.Accuracy(task="binary").to(device)

In [27]:
history_B = train(model_B, optimizer, xentropy, accuracy,
                  train_loader_B, valid_loader_B, n_epochs=20)
history_B

Epoch 1/20 | loss: 0.6917 | train acc: 0.6000 | val acc: 0.5147
Epoch 2/20 | loss: 0.6313 | train acc: 0.8000 | val acc: 0.5058
Epoch 3/20 | loss: 0.6024 | train acc: 0.7000 | val acc: 0.5070
Epoch 4/20 | loss: 0.5814 | train acc: 0.6500 | val acc: 0.5060
Epoch 5/20 | loss: 0.5646 | train acc: 0.6500 | val acc: 0.5060
Epoch 6/20 | loss: 0.5498 | train acc: 0.6500 | val acc: 0.5062
Epoch 7/20 | loss: 0.5367 | train acc: 0.7000 | val acc: 0.5072
Epoch 8/20 | loss: 0.5248 | train acc: 0.7000 | val acc: 0.5074
Epoch 9/20 | loss: 0.5157 | train acc: 0.7000 | val acc: 0.5084
Epoch 10/20 | loss: 0.5065 | train acc: 0.7500 | val acc: 0.5088
Epoch 11/20 | loss: 0.4980 | train acc: 0.7500 | val acc: 0.5088
Epoch 12/20 | loss: 0.4902 | train acc: 0.7500 | val acc: 0.5094
Epoch 13/20 | loss: 0.4823 | train acc: 0.7500 | val acc: 0.5086
Epoch 14/20 | loss: 0.4753 | train acc: 0.7500 | val acc: 0.5088
Epoch 15/20 | loss: 0.4678 | train acc: 0.7500 | val acc: 0.5080
Epoch 16/20 | loss: 0.4602 | train

{'train_losses': [0.6917290687561035,
  0.6313420534133911,
  0.6024067401885986,
  0.5813739895820618,
  0.564556896686554,
  0.5498436093330383,
  0.5366654992103577,
  0.5248367786407471,
  0.5156525373458862,
  0.5064703226089478,
  0.49798235297203064,
  0.49015116691589355,
  0.4823237359523773,
  0.47530150413513184,
  0.4677920341491699,
  0.46016988158226013,
  0.45315027236938477,
  0.446460098028183,
  0.4390426576137543,
  0.4314902424812317],
 'train_metrics': [0.6000000238418579,
  0.800000011920929,
  0.699999988079071,
  0.6499999761581421,
  0.6499999761581421,
  0.6499999761581421,
  0.699999988079071,
  0.699999988079071,
  0.699999988079071,
  0.75,
  0.75,
  0.75,
  0.75,
  0.75,
  0.75,
  0.75,
  0.800000011920929,
  0.800000011920929,
  0.800000011920929,
  0.800000011920929],
 'valid_metrics': [0.5146586298942566,
  0.5058233141899109,
  0.5070281028747559,
  0.5060241222381592,
  0.5060241222381592,
  0.5062248706817627,
  0.5072289109230042,
  0.50742971897125

In [28]:
evaluate_tm(model_B, test_loader_B, accuracy)

tensor(0.4881)

In [29]:
import copy

torch.manual_seed(43)
reused_layers = copy.deepcopy(model_A[:-1])
model_B_on_A = nn.Sequential(
    *reused_layers,
    nn.Linear(64, 1)  # new output layer for task B
).to(device)

In [30]:
for layer in model_B_on_A[:-1]:
    for param in layer.parameters():
        param.requires_grad = False

In [31]:
n_epochs = 10
optimizer = torch.optim.SGD(model_B_on_A.parameters(), lr=0.005)
xentropy = nn.BCEWithLogitsLoss()
accuracy = torchmetrics.Accuracy(task="binary").to(device)
history_B = train(model_B_on_A, optimizer, xentropy, accuracy,
                  train_loader_B, valid_loader_B, n_epochs)

Epoch 1/10 | loss: 0.7815 | train acc: 0.4000 | val acc: 0.4982
Epoch 2/10 | loss: 0.7649 | train acc: 0.3500 | val acc: 0.5030
Epoch 3/10 | loss: 0.7500 | train acc: 0.3500 | val acc: 0.5008
Epoch 4/10 | loss: 0.7366 | train acc: 0.4000 | val acc: 0.5020
Epoch 5/10 | loss: 0.7245 | train acc: 0.5000 | val acc: 0.5004
Epoch 6/10 | loss: 0.7137 | train acc: 0.4500 | val acc: 0.5036
Epoch 7/10 | loss: 0.7040 | train acc: 0.5500 | val acc: 0.5060
Epoch 8/10 | loss: 0.6953 | train acc: 0.5500 | val acc: 0.5020
Epoch 9/10 | loss: 0.6874 | train acc: 0.5500 | val acc: 0.5032
Epoch 10/10 | loss: 0.6804 | train acc: 0.5500 | val acc: 0.5070


In [32]:
for layer in model_B_on_A[2:]:
    for param in layer.parameters():
        param.requires_grad = True

In [33]:
n_epochs = 20
optimizer = torch.optim.SGD(model_B_on_A.parameters(), lr=0.005)
xentropy = nn.BCEWithLogitsLoss()
accuracy = torchmetrics.Accuracy(task="binary").to(device)
history_B = train(model_B_on_A, optimizer, xentropy, accuracy,
                  train_loader_B, valid_loader_B, n_epochs)

Epoch 1/20 | loss: 0.6740 | train acc: 0.5500 | val acc: 0.5020
Epoch 2/20 | loss: 0.6661 | train acc: 0.6000 | val acc: 0.5040
Epoch 3/20 | loss: 0.6591 | train acc: 0.5000 | val acc: 0.5048
Epoch 4/20 | loss: 0.6529 | train acc: 0.5500 | val acc: 0.5030
Epoch 5/20 | loss: 0.6473 | train acc: 0.5500 | val acc: 0.5042
Epoch 6/20 | loss: 0.6423 | train acc: 0.6000 | val acc: 0.5032
Epoch 7/20 | loss: 0.6377 | train acc: 0.6500 | val acc: 0.4996
Epoch 8/20 | loss: 0.6336 | train acc: 0.6500 | val acc: 0.5006
Epoch 9/20 | loss: 0.6298 | train acc: 0.6500 | val acc: 0.4994
Epoch 10/20 | loss: 0.6263 | train acc: 0.6500 | val acc: 0.5006
Epoch 11/20 | loss: 0.6231 | train acc: 0.6500 | val acc: 0.5014
Epoch 12/20 | loss: 0.6202 | train acc: 0.6500 | val acc: 0.5026
Epoch 13/20 | loss: 0.6174 | train acc: 0.7000 | val acc: 0.5062
Epoch 14/20 | loss: 0.6148 | train acc: 0.7000 | val acc: 0.5070
Epoch 15/20 | loss: 0.6124 | train acc: 0.7000 | val acc: 0.5088
Epoch 16/20 | loss: 0.6101 | train

In [34]:
evaluate_tm(model_B_on_A, test_loader_B, accuracy)

tensor(0.4949)

# Optimizers

In [35]:
import matplotlib.pyplot as plt

In [40]:
def build_model(seed=42):
  torch.manual_seed(seed)
  model = nn.Sequential(
      nn.Flatten(),
      nn.Linear(3*32*32, 256),
      nn.ReLU(),
      nn.Linear(256, 128),
      nn.ReLU(),
      nn.Linear(128, 64),
      nn.ReLU(),
      nn.Linear(64, 10)
  ).to(device)
  model.apply(use_he_init)
  return model

In [41]:
train_set = TensorDataset(X_train[:45_000], y_train[:45_000])
valid_set = TensorDataset(X_train[45_000:], y_train[45_000:])
test_set = TensorDataset(X_test, y_test)

In [42]:
def test_optimizer(model, optimizer, n_epochs=10, batch_size=32):
    train_loader = DataLoader(train_set, batch_size=batch_size, shuffle=True)
    valid_loader = DataLoader(valid_set, batch_size=batch_size)
    test_loader  = DataLoader(test_set,  batch_size=batch_size)
    xentropy = nn.CrossEntropyLoss()
    accuracy = torchmetrics.Accuracy(task="multiclass", num_classes=10).to(device)
    history  = train(model, optimizer, xentropy, accuracy,
                     train_loader, valid_loader, n_epochs)
    test_acc = evaluate_tm(model, test_loader, accuracy)
    return history, test_acc

# SGD

In [43]:
model = build_model()
optimizer = torch.optim.SGD(model.parameters(), lr=0.05)
history_sgd, acc_sgd = test_optimizer(model, optimizer)

Epoch 1/10 | loss: 1.9154 | train acc: 0.2999 | val acc: 0.2968
Epoch 2/10 | loss: 1.7220 | train acc: 0.3790 | val acc: 0.2664
Epoch 3/10 | loss: 1.6366 | train acc: 0.4120 | val acc: 0.3866
Epoch 4/10 | loss: 1.5820 | train acc: 0.4340 | val acc: 0.3678
Epoch 5/10 | loss: 1.5373 | train acc: 0.4485 | val acc: 0.4294
Epoch 6/10 | loss: 1.5000 | train acc: 0.4637 | val acc: 0.4126
Epoch 7/10 | loss: 1.4735 | train acc: 0.4702 | val acc: 0.4018
Epoch 8/10 | loss: 1.4438 | train acc: 0.4805 | val acc: 0.4472
Epoch 9/10 | loss: 1.4200 | train acc: 0.4914 | val acc: 0.4476
Epoch 10/10 | loss: 1.3948 | train acc: 0.5005 | val acc: 0.4186


# Momentum

In [44]:
model = build_model()
optimizer = torch.optim.SGD(model.parameters(), momentum=0.9, lr=0.05)
history_momentum, acc_momentum = test_optimizer(model, optimizer)

Epoch 1/10 | loss: 2.3120 | train acc: 0.1017 | val acc: 0.1058
Epoch 2/10 | loss: 2.3064 | train acc: 0.0976 | val acc: 0.0970
Epoch 3/10 | loss: 2.3063 | train acc: 0.0991 | val acc: 0.0950
Epoch 4/10 | loss: 2.3066 | train acc: 0.0986 | val acc: 0.0970
Epoch 5/10 | loss: 2.3062 | train acc: 0.1001 | val acc: 0.1038
Epoch 6/10 | loss: 2.3065 | train acc: 0.0980 | val acc: 0.0958
Epoch 7/10 | loss: 2.3063 | train acc: 0.1021 | val acc: 0.0976
Epoch 8/10 | loss: 2.3067 | train acc: 0.1007 | val acc: 0.0950
Epoch 9/10 | loss: 2.3059 | train acc: 0.1006 | val acc: 0.1064
Epoch 10/10 | loss: 2.3065 | train acc: 0.1006 | val acc: 0.0950


# Nesterov

In [45]:
model = build_model()
optimizer = torch.optim.SGD(model.parameters(), momentum=0.9, nesterov=True, lr=0.05)
history_nesterov, acc_nesterov = test_optimizer(model, optimizer)

Epoch 1/10 | loss: 2.3304 | train acc: 0.1019 | val acc: 0.1058
Epoch 2/10 | loss: 2.3063 | train acc: 0.0976 | val acc: 0.0970
Epoch 3/10 | loss: 2.3061 | train acc: 0.0983 | val acc: 0.0950
Epoch 4/10 | loss: 2.3064 | train acc: 0.0992 | val acc: 0.0950
Epoch 5/10 | loss: 2.3061 | train acc: 0.1006 | val acc: 0.1038
Epoch 6/10 | loss: 2.3063 | train acc: 0.0987 | val acc: 0.0958
Epoch 7/10 | loss: 2.3062 | train acc: 0.1020 | val acc: 0.1058
Epoch 8/10 | loss: 2.3065 | train acc: 0.1010 | val acc: 0.0950
Epoch 9/10 | loss: 2.3058 | train acc: 0.1009 | val acc: 0.1064
Epoch 10/10 | loss: 2.3063 | train acc: 0.1004 | val acc: 0.0950


# AdaGrad

In [46]:
model = build_model()
optimizer = torch.optim.Adagrad(model.parameters(), lr=0.05)
history_adagrad, acc_adagrad = test_optimizer(model, optimizer)

Epoch 1/10 | loss: 3.0013 | train acc: 0.2495 | val acc: 0.2836
Epoch 2/10 | loss: 1.8229 | train acc: 0.3404 | val acc: 0.2938
Epoch 3/10 | loss: 1.7479 | train acc: 0.3680 | val acc: 0.3484
Epoch 4/10 | loss: 1.7060 | train acc: 0.3834 | val acc: 0.3512
Epoch 5/10 | loss: 1.6732 | train acc: 0.3981 | val acc: 0.3902
Epoch 6/10 | loss: 1.6470 | train acc: 0.4094 | val acc: 0.3980
Epoch 7/10 | loss: 1.6238 | train acc: 0.4179 | val acc: 0.3986
Epoch 8/10 | loss: 1.6010 | train acc: 0.4270 | val acc: 0.4260
Epoch 9/10 | loss: 1.5754 | train acc: 0.4337 | val acc: 0.4238
Epoch 10/10 | loss: 1.5556 | train acc: 0.4443 | val acc: 0.4022


# RMSProp

In [47]:
model = build_model()
optimizer = torch.optim.RMSprop(model.parameters(), alpha=0.9, lr=0.05)
history_rmsprop, acc_rmsprop = test_optimizer(model, optimizer)

Epoch 1/10 | loss: 28.3403 | train acc: 0.1031 | val acc: 0.1058
Epoch 2/10 | loss: 2.3096 | train acc: 0.1014 | val acc: 0.0970
Epoch 3/10 | loss: 2.3088 | train acc: 0.1003 | val acc: 0.1024
Epoch 4/10 | loss: 2.3090 | train acc: 0.1015 | val acc: 0.1034
Epoch 5/10 | loss: 2.3095 | train acc: 0.1002 | val acc: 0.1034
Epoch 6/10 | loss: 2.3091 | train acc: 0.0994 | val acc: 0.1058
Epoch 7/10 | loss: 2.3097 | train acc: 0.1004 | val acc: 0.1058
Epoch 8/10 | loss: 2.3092 | train acc: 0.1009 | val acc: 0.0958
Epoch 9/10 | loss: 2.3093 | train acc: 0.0995 | val acc: 0.1064
Epoch 10/10 | loss: 2.3090 | train acc: 0.1013 | val acc: 0.0950


# Adam

In [ ]:
model = build_model()
optimizer = torch.optim.Adam(model.parameters(), betas=(0.9, 0.999), lr=0.05)
history_adam, acc_adam = test_optimizer(model, optimizer)

Epoch 1/10 | loss: 3.3043 | train acc: 0.1019 | val acc: 0.1058


# Adamax

In [ ]:
model = build_model()
optimizer = torch.optim.Adamax(model.parameters(), betas=(0.9, 0.999), lr=0.05)
history_adamax, acc_adamax = test_optimizer(model, optimizer)

# Nadam

In [ ]:
model = build_model()
optimizer = torch.optim.NAdam(model.parameters(), betas=(0.9, 0.999), lr=0.05)
history_nadam, acc_nadam = test_optimizer(model, optimizer)

# AdamW

In [ ]:
model = build_model()
optimizer = torch.optim.AdamW(model.parameters(), betas=(0.9, 0.999),
                               weight_decay=1e-5, lr=0.05)
history_adamw, acc_adamw = test_optimizer(model, optimizer)

In [ ]:
opt_names = ["SGD", "Momentum", "Nesterov", "AdaGrad", "RMSProp",
             "Adam", "Adamax", "Nadam", "AdamW"]
accs = [acc_sgd, acc_momentum, acc_nesterov, acc_adagrad, acc_rmsprop,
        acc_adam, acc_adamax, acc_nadam, acc_adamw]

In [ ]:
print("\n=== Test Accuracy ===")
for name, acc in zip(opt_names, accs):
    print(f"{name:12s}: {acc:.4f}")

In [ ]:
# Vizualizasiya
histories = [history_sgd, history_momentum, history_nesterov, history_adagrad,
             history_rmsprop, history_adam, history_adamax, history_nadam,
             history_adamw]

In [ ]:
for plot in ("train_losses", "valid_metrics"):
    plt.figure(figsize=(10, 5))
    for history, name in zip(histories, opt_names):
        plt.plot(history[plot], label=name, linewidth=2)
    plt.grid()
    plt.xlabel("Epochs")
    plt.ylabel("Training Loss" if plot == "train_losses" else "Validation Accuracy")
    plt.title(f"CIFAR-10 — {'Training Loss' if plot == 'train_losses' else 'Validation Accuracy'}")
    plt.legend(loc="upper right")
    plt.tight_layout()
    plt.show()